In [24]:
import pandas as pd

# Load the cleaned dataset
df = pd.read_csv('./outputs/Global_Graphite_Projects_final.csv')
print(f"Loaded {len(df)} records across {df['Country (Short Form)'].nunique()} countries")

Loaded 248 records across 33 countries


In [25]:
# USGS Mineral Commodity Summaries: Graphite production by country
# Source: MCS 2019 (published Feb 2019, data for 2017-2018)
# https://d9-wret.s3-us-west-2.amazonaws.com/assets/palladium/production/s3fs-public/atoms/files/mcs-2019-graph.pdf
# Source: MCS 2026 (published Feb 2026, data for 2024-2025)
# https://pubs.usgs.gov/periodicals/mcs2026/mcs2026-graphite.pdf


usgs_production = pd.DataFrame([
    # Country                    2017       2018     2024        2025     Reserves
    ('Brazil',                  90_000,    95_000,   58_000,    65_000,  74_000_000),
    ('Canada',                  40_000,    40_000,   11_700,     8_000,   5_900_000),
    ('China',                  625_000,   630_000, 1_270_000, 1_400_000, 100_000_000),
    ('India',                   35_000,    35_000,   17_600,    17_000,   8_600_000),
    ('Korea, North',             5_500,     6_000,    8_100,     8_000,   2_000_000),
    ('Madagascar',               9_000,     9_000,   85_000,    80_000,  27_000_000),
    ('Mexico',                   9_000,     9_000,      706,       740,   3_100_000),
    ('Mozambique',                 300,    20_000,   39_000,    60_000,  25_000_000),
    ('Namibia',                  2_220,     2_200,     None,      None,       None),
    ('Norway',                  15_500,    16_000,    5_340,     6_600,     600_000),
    ('Pakistan',                14_000,    14_000,     None,      None,       None),
    ('Russia',                  17_000,    17_000,   20_000,    25_000,  14_000_000),
    ('Sri Lanka',                3_500,     4_000,    3_000,     3_200,   1_500_000),
    ('Tanzania',                  None,      None,   27_000,    75_000,  18_000_000),
    ('Turkey',                   2_300,     2_000,    2_600,     2_200,   6_900_000),
    ('Ukraine',                 20_000,    20_000,      900,       800,       None),
    ('Vietnam',                  5_000,     5_000,      500,       500,   9_700_000),
    ('Zimbabwe',                 1_580,     2_000,     None,      None,       None),
    # New in MCS 2026 (not in 2019)
    ('Austria',                   None,      None,      100,       200,       None),
    ('Germany',                   None,      None,      140,       140,       None),
    ('Korea, South',              None,      None,    1_000,       500,   1_800_000),
], columns=['Country', 'Production_2017', 'Production_2018', 
            'Production_2024', 'Production_2025', 'Reserves'])

# World totals from the MCS reports
usgs_world_totals = {
    2017: 897_000,
    2018: 930_000,
    2024: 1_550_000,
    2025: 1_800_000,
}

print(f"USGS reference: {len(usgs_production)} producing countries")
usgs_production

USGS reference: 21 producing countries


,Country,Production_2017,Production_2018,Production_2024,Production_2025,Reserves
0,Brazil,90000.0,95000.0,58000.0,65000.0,74000000.0
1,Canada,40000.0,40000.0,11700.0,8000.0,5900000.0
2,China,625000.0,630000.0,1270000.0,1400000.0,100000000.0
3,India,35000.0,35000.0,17600.0,17000.0,8600000.0
4,"Korea, North",5500.0,6000.0,8100.0,8000.0,2000000.0
5,Madagascar,9000.0,9000.0,85000.0,80000.0,27000000.0
6,Mexico,9000.0,9000.0,706.0,740.0,3100000.0
7,Mozambique,300.0,20000.0,39000.0,60000.0,25000000.0
8,Namibia,2220.0,2200.0,NaN,NaN,NaN
9,Norway,15500.0,16000.0,5340.0,6600.0,600000.0


In [26]:
# Standardise country names in dataset to match USGS
country_map = {
    'North Korea': 'Korea, North',
    'Korea, Republic of': 'Korea, South',
    'South Korea': 'Korea, South',
}
df['Country_std'] = df['Country (Short Form)'].replace(country_map)

# Build a summary of our data by country
# Count records by type
our_summary = df.groupby('Country_std').agg(
    total_records=('Name', 'count'),
    facilities=('Source_Layer', lambda x: x.str.contains('Facilit', case=False).sum()),
    deposits=('Source_Layer', lambda x: x.str.contains('Deposit', case=False).sum()),
    exploration=('Source_Layer', lambda x: x.str.contains('Explor', case=False).sum()),
    development=('Source_Layer', lambda x: x.str.contains('Develop', case=False).sum()),
    # Sum capacity (excluding -999 sentinel values)
    total_capacity_mt=('Annual Production Capacity', 
                       lambda x: x[x > 0].sum() if (x > 0).any() else 0),
    facilities_with_capacity=('Annual Production Capacity', 
                              lambda x: ((x.notna()) & (x > 0)).sum()),
).reset_index().rename(columns={'Country_std': 'Country'})

print(f"Our dataset covers {len(our_summary)} countries")
our_summary.sort_values('total_records', ascending=False)

Our dataset covers 32 countries


,Country,total_records,facilities,deposits,exploration,development,total_capacity_mt,facilities_with_capacity
24,Sri Lanka,31,3,0,27,1,8280.0,3
21,Norway,28,0,0,0,0,0.0,0
6,China,27,4,11,6,0,568000.0,7
16,Madagascar,27,4,14,9,0,169000.0,4
1,Australia,18,0,14,0,0,0.0,0
10,India,17,0,12,2,0,63000.0,2
26,Tanzania,14,0,0,14,0,0.0,0
4,Canada,11,0,0,0,0,0.0,0
14,"Korea, South",8,0,0,8,0,0.0,0
13,"Korea, North",7,0,2,0,0,6000.0,3


In [27]:
# Merge data with USGS production data
coverage = usgs_production.merge(our_summary, on='Country', how='outer', indicator=True)

# Classify coverage
def classify(row):
    if row['_merge'] == 'left_only':
        return '[X] USGS producer - NOT in data'
    elif row['_merge'] == 'right_only':
        return '[+] In our data - NOT a USGS listed producer'
    else:
        return '[✓] Covered'

coverage['Coverage'] = coverage.apply(classify, axis=1)
coverage = coverage.sort_values(['Coverage', 'Production_2025'], ascending=[True, False])

# Display
print("=" * 80)
print("COVERAGE ANALYSIS: Our Dataset vs USGS Producing Countries")
print("=" * 80)
print()

for status in ['[✓] Covered', '[X] USGS producer - NOT in data', '[+] In our data - NOT a USGS listed producer']:
    subset = coverage[coverage['Coverage'] == status]
    print(f"\n{status} ({len(subset)} countries):")
    print("-" * 60)
    if status == '[✓] Covered':
        cols = ['Country', 'Production_2025', 'total_records', 'facilities', 
                'deposits', 'exploration', 'total_capacity_mt']
        print(subset[cols].to_string(index=False))
    elif status == '[X] USGS producer - NOT in data':
        cols = ['Country', 'Production_2018', 'Production_2025']
        print(subset[cols].to_string(index=False))
    else:
        cols = ['Country', 'total_records', 'facilities', 'deposits', 'exploration']
        print(subset[cols].to_string(index=False))

# Summary stats
n_usgs = len(usgs_production)
n_covered = len(coverage[coverage['Coverage'] == '[✓] Covered'])
n_missing = len(coverage[coverage['Coverage'] == '[X] USGS producer - NOT in data'])
print(f"\n\nSUMMARY: {n_covered}/{n_usgs} USGS producing countries covered ({100*n_covered/n_usgs:.0f}%)")
print(f"Missing: {n_missing} countries")

COVERAGE ANALYSIS: Our Dataset vs USGS Producing Countries


[✓] Covered (17 countries):
------------------------------------------------------------
     Country  Production_2025  total_records  facilities  deposits  exploration  total_capacity_mt
       China        1400000.0           27.0         4.0      11.0          6.0           568000.0
  Madagascar          80000.0           27.0         4.0      14.0          9.0           169000.0
    Tanzania          75000.0           14.0         0.0       0.0         14.0                0.0
      Brazil          65000.0            3.0         0.0       0.0          0.0             4000.0
  Mozambique          60000.0            7.0         2.0       0.0          5.0             9350.0
      Russia          25000.0            1.0         0.0       0.0          0.0                0.0
       India          17000.0           17.0         0.0      12.0          2.0            63000.0
      Canada           8000.0           11.0         0.0  

In [28]:
# Production coverage: what % of world production do we capture?
covered_countries = coverage[coverage['Coverage'] == '[✓] Covered']

for year, col in [(2018, 'Production_2018'), (2025, 'Production_2025')]:
    covered_prod = covered_countries[col].sum()
    world_total = usgs_world_totals[year]
    pct = 100 * covered_prod / world_total
    print(f"\n{year} Production Coverage:")
    print(f"  Countries we cover produced: {covered_prod:,.0f} mt")
    print(f"  World total:                 {world_total:,.0f} mt")
    print(f"  Coverage:                    {pct:.1f}%")
    
    # What are we missing?
    missing = coverage[(coverage['Coverage'] == '[X] USGS producer - NOT in data') & coverage[col].notna()]
    if not missing.empty:
        missing_prod = missing[col].sum()
        print(f"  Missing production:          {missing_prod:,.0f} mt ({100*missing_prod/world_total:.1f}%)")
        print(f"  Missing countries:           {', '.join(missing['Country'].tolist())}")


2018 Production Coverage:
  Countries we cover produced: 910,200 mt
  World total:                 930,000 mt
  Coverage:                    97.9%
  Missing production:          16,000 mt (1.7%)
  Missing countries:           Turkey, Pakistan

2025 Production Coverage:
  Countries we cover produced: 1,750,340 mt
  World total:                 1,800,000 mt
  Coverage:                    97.2%
  Missing production:          2,540 mt (0.1%)
  Missing countries:           Turkey, Austria, Germany


In [29]:
# Capacity vs Production comparison (where we have both)
# This shows whether our capacity data is in the right ballpark
compare = covered_countries[
    (covered_countries['total_capacity_mt'] > 0) & 
    (covered_countries['Production_2018'].notna())
][['Country', 'total_capacity_mt', 'Production_2018', 'Production_2025', 'facilities_with_capacity']].copy()

if not compare.empty:
    compare['Capacity_vs_Prod_2018'] = (compare['total_capacity_mt'] / compare['Production_2018'] * 100).round(0)
    compare.columns = ['Country', 'Our Capacity (mt)', 'USGS Prod 2018', 'USGS Prod 2025', 
                        '# Facilities', 'Capacity/Prod 2018 (%)']
    
    print("\nCapacity vs Production Comparison:")
    print("(Capacity/Production > 100% = we may have overcounted or capacity > actual production)")
    print("(Capacity/Production < 100% = we're likely missing facilities)")
    print("-" * 90)
    print(compare.to_string(index=False))
else:
    print("No countries with both capacity and production data to compare.")


Capacity vs Production Comparison:
(Capacity/Production > 100% = we may have overcounted or capacity > actual production)
(Capacity/Production < 100% = we're likely missing facilities)
------------------------------------------------------------------------------------------
     Country  Our Capacity (mt)  USGS Prod 2018  USGS Prod 2025  # Facilities  Capacity/Prod 2018 (%)
       China           568000.0        630000.0       1400000.0           7.0                    90.0
  Madagascar           169000.0          9000.0         80000.0           4.0                  1878.0
      Brazil             4000.0         95000.0         65000.0           2.0                     4.0
  Mozambique             9350.0         20000.0         60000.0           2.0                    47.0
       India            63000.0         35000.0         17000.0           2.0                   180.0
Korea, North             6000.0          6000.0          8000.0           3.0                   100.0
   Sri La

In [30]:
# Data freshness analysis
print("\nData Freshness by Region:")
print("-" * 50)
for region in df['Region'].unique():
    subset = df[df['Region'] == region]
    ref_years = subset['Reference Year'].dropna()
    if not ref_years.empty:
        print(f"  {region}: Reference years {int(ref_years.min())}-{int(ref_years.max())} "
              f"(median {int(ref_years.median())})")
    else:
        print(f"  {region}: No reference year data")

print("\n[!]  Note: USGS MCS 2019 reflects 2017-2018 data; MCS 2026 reflects 2024-2025 data.")
print("   Our dataset's reference years indicate when each record was last updated.")
print("   Facilities/exploration sites may have changed status since the reference year.")


Data Freshness by Region:
--------------------------------------------------
  Africa: Reference years 2009-2021 (median 2016)
  China: Reference years 2017-2023 (median 2023)
  Indopacific: Reference years 2014-2023 (median 2021)
  SW Asia: Reference years 2018-2018 (median 2018)
  Latin America & Caribbean: Reference years 2014-2020 (median 2014)
  Australia: Reference years 2015-2024 (median 2023)
  North America: Reference years 2012-2023 (median 2018)
  Europe: Reference years 2017-2023 (median 2020)
  Europe/Asia: Reference years 2017-2017 (median 2017)

[!]  Note: USGS MCS 2019 reflects 2017-2018 data; MCS 2026 reflects 2024-2025 data.
   Our dataset's reference years indicate when each record was last updated.
   Facilities/exploration sites may have changed status since the reference year.
